# Kaggle runner: MIO-TCD notebooks 04–06

Single entry point for Kaggle. Attach two private Kaggle datasets before running: (1) raw MIO-TCD containing `train/` and `gt_train.csv`; (2) prepared split containing `split_manifest.csv`. The runner never uses original MIO `test/`, never changes the frozen split, and writes only below `/kaggle/working`.

## 1. Config — edit these paths

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Thudoanh/area-violation-detection.git'
PROJECT_ROOT = Path('/kaggle/working/area-violation-detection')

# Must directly contain train/ and gt_train.csv.
MIO_TCD_ROOT = Path('/kaggle/input/CHANGE_ME_MIO_DATASET/MIO-TCD-Localization')

# Must directly contain split_manifest.csv (plus split_config.yaml if available).
SPLIT_SOURCE_DIR = Path('/kaggle/input/CHANGE_ME_PREPARED_DATASET/splits')

RUN_BASELINE = True
RUN_FINETUNE = False  # Turn on after reviewing baseline output
RUN_FINETUNED_EVAL = False  # Turn on after fine-tuning finishes
REBUILD_YOLO_DATA = True

## 2. Validate Kaggle runtime and attached inputs

In [ ]:
import os, sys, shutil, subprocess
import torch

if not MIO_TCD_ROOT.is_dir():
    raise FileNotFoundError(f'Edit MIO_TCD_ROOT; directory not found: {MIO_TCD_ROOT}')
if not (MIO_TCD_ROOT / 'train').is_dir():
    raise FileNotFoundError(f'Missing raw train directory: {MIO_TCD_ROOT / "train"}')
if not (MIO_TCD_ROOT / 'gt_train.csv').is_file():
    raise FileNotFoundError(f'Missing ground truth: {MIO_TCD_ROOT / "gt_train.csv"}')
if not (SPLIT_SOURCE_DIR / 'split_manifest.csv').is_file():
    raise FileNotFoundError(f'Missing frozen manifest: {SPLIT_SOURCE_DIR / "split_manifest.csv"}')

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle before training.')

## 3. Clone source and install dependencies

In [ ]:
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    print('Updated existing clone:', PROJECT_ROOT)
elif PROJECT_ROOT.exists():
    raise FileExistsError(f'{PROJECT_ROOT} exists but is not a Git clone; remove only this Kaggle working copy and retry.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
    check=True,
)
print('Source and dependencies ready.')

## 4. Copy and relocate the frozen split

In [ ]:
import pandas as pd
from pathlib import PureWindowsPath
from tqdm.auto import tqdm

split_target = PROJECT_ROOT / 'data/mio_tcd/splits'
split_target.mkdir(parents=True, exist_ok=True)
for name in ('split_manifest.csv', 'split_config.yaml', 'excluded_invalid_images.csv'):
    source = SPLIT_SOURCE_DIR / name
    if source.is_file():
        shutil.copy2(source, split_target / name)

manifest_path = split_target / 'split_manifest.csv'
manifest = pd.read_csv(manifest_path, dtype={'image_id': str})
if 'image_path' in manifest.columns:
    original_ids = manifest['image_id'].copy()
    recovered_ids = manifest['image_path'].astype(str).map(lambda value: PureWindowsPath(value).stem)
    if recovered_ids.ne('').all() and recovered_ids.is_unique:
        manifest['image_id'] = recovered_ids
        print('Recovered leading-zero image IDs:', int(original_ids.ne(recovered_ids).sum()))
required = {'image_id', 'split', 'width', 'height'}
if required - set(manifest.columns):
    raise ValueError(f'Manifest lacks {sorted(required - set(manifest.columns))}; rerun local notebook 02 and upload it again.')
if manifest.image_id.duplicated().any() or set(manifest.split) - {'train', 'val', 'test'}:
    raise ValueError('Frozen split manifest is invalid.')

def cloud_image_path(image_id):
    for suffix in ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP'):
        candidate = MIO_TCD_ROOT / 'train' / f'{image_id}{suffix}'
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(f'Raw image not found for image_id={image_id}')

manifest['image_path'] = [cloud_image_path(image_id) for image_id in tqdm(manifest.image_id, desc='Relocating image paths')]
manifest.to_csv(manifest_path, index=False)
for split in ('train', 'val', 'test'):
    values = manifest.loc[manifest.split.eq(split), 'image_path']
    (split_target / f'{split}.txt').write_text('\n'.join(values) + '\n', encoding='utf-8')
print(manifest.groupby('split').size())

## 5. Build Kaggle-local YOLO labels and image views

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
os.environ['MIO_TCD_ROOT'] = str(MIO_TCD_ROOT)

from scripts import mio_tcd_utils as mio_utils
mio_utils.PROJECT_ROOT = PROJECT_ROOT

annotations = mio_utils.read_annotations(MIO_TCD_ROOT)
yolo_root = mio_utils.prepare_yolo(annotations, manifest, force=REBUILD_YOLO_DATA)
print('YOLO config:', yolo_root / 'mio_tcd.yaml')
print((yolo_root / 'mio_tcd.yaml').read_text())

## 6. Execute notebooks 04–06

In [ ]:
executed_dir = Path('/kaggle/working/executed-notebooks')
executed_dir.mkdir(parents=True, exist_ok=True)

def execute_notebook(filename):
    source = PROJECT_ROOT / 'scripts' / filename
    output = executed_dir / filename
    if not source.is_file():
        raise FileNotFoundError(source)
    print(f'Executing {filename} ...')
    command = ['jupyter', 'nbconvert', '--to', 'notebook', '--execute', str(source),
               '--output', filename, '--output-dir', str(executed_dir),
               '--ExecutePreprocessor.timeout=-1']
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, capture_output=True, text=True,
        env={**os.environ, 'MIO_TCD_ROOT': str(MIO_TCD_ROOT)},
    )
    log_path = executed_dir / f'{Path(filename).stem}.log'
    log_path.write_text(completed.stdout + '\n' + completed.stderr, encoding='utf-8')
    if completed.stdout.strip(): print(completed.stdout)
    if completed.stderr.strip(): print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'{filename} failed. Full nbconvert log: {log_path}')
    print('Completed:', output)

if RUN_BASELINE:
    baseline_run = PROJECT_ROOT / 'outputs/mio_tcd/baseline/yolo11n_pretrained_test'
    baseline_summary = PROJECT_ROOT / 'outputs/mio_tcd/baseline_metrics.csv'
    if baseline_run.exists() and not baseline_summary.is_file():
        shutil.rmtree(baseline_run)
        print('Removed incomplete baseline run:', baseline_run)
    execute_notebook('04_yolo11_baseline_eval.ipynb')
    baseline_path = PROJECT_ROOT / 'outputs/mio_tcd/baseline_metrics.csv'
    per_class_path = PROJECT_ROOT / 'outputs/mio_tcd/baseline_per_class_metrics.csv'
    print('YOLO11n pretrained overall metrics:')
    display(pd.read_csv(baseline_path))
    print('YOLO11n pretrained per-class metrics:')
    display(pd.read_csv(per_class_path))
if RUN_FINETUNE:
    execute_notebook('05_yolo11_finetune.ipynb')
if RUN_FINETUNED_EVAL:
    execute_notebook('06_yolo11_finetuned_eval.ipynb')

## 7. Validate and package outputs

In [ ]:
outputs = PROJECT_ROOT / 'outputs/mio_tcd'
expected = []
if RUN_BASELINE: expected.append(outputs / 'baseline_metrics.csv')
if RUN_FINETUNE: expected.append(outputs / 'train/yolo11n_mio_v1/weights/best.pt')
if RUN_FINETUNED_EVAL: expected.extend([outputs / 'finetuned_metrics.csv', outputs / 'model_comparison.csv'])
missing = [str(path) for path in expected if not path.is_file()]
if missing:
    raise FileNotFoundError('Expected outputs are missing: ' + ', '.join(missing))

archive = shutil.make_archive('/kaggle/working/mio_tcd_outputs', 'zip', root_dir=outputs)
print('Download or save this Kaggle output:', archive)